# ✦ LUMINA — Fine-tune LoRA trên Kaggle (GPU FREE)

Notebook chạy **QLoRA (4-bit)** trên GPU miễn phí của Kaggle (2×T4 16GB hoặc P100)
để tạo **LoRA adapter** cho bộ não local của LUMINA. Xuất ra file adapter (thường
**50–400MB**) tải về được, rồi gộp + convert GGUF cho Ollama (xem `export_ollama.md`).

## Sự thật cần biết TRƯỚC khi chạy (đọc kỹ)

- **Kaggle free ≈ chỉ fine-tune nổi model ≤ 7–8B** (QLoRA). 14B rất sát giới hạn,
  **22B thì KHÔNG** train nổi trên Kaggle free (thiếu VRAM). Mặc định notebook dùng
  `Qwen/Qwen2.5-7B-Instruct` — an toàn.
- **File LoRA < 400MB là ADAPTER, KHÔNG phải model đứng một mình.** Khi chạy vẫn cần
  nạp model gốc (7B) kèm adapter. Không có chuyện nhét cả 22B vào < 400MB (xem ô cuối).
- **Bake skills vào trọng số thường KÉM HƠN** tiêm prompt lúc chạy (LUMINA đang làm).
  Chỉ nên fine-tune để dạy **văn phong / định dạng / hành vi**, không phải nhồi kiến thức.
- Bật GPU: **Settings → Accelerator → GPU T4 x2** (hoặc P100). Bật **Internet** để tải model/dataset.


## 1) Cài thư viện


In [ ]:
!pip -q install -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" \
  "datasets>=2.20" "bitsandbytes>=0.43" accelerate sentencepiece 2>/dev/null
import torch, os
print('CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only!')


## 2) Dữ liệu train (chat JSONL: mỗi dòng `{"messages":[...]}`)

**Cách A — dùng dataset SFT thật** (khuyến nghị): tải trực tiếp từ HuggingFace.  
**Cách B — bake skills LUMINA**: upload thư mục `data/skills/` thành *Kaggle Dataset*
rồi trỏ `SKILLS_DIR` vào đó (không khuyến nghị làm dữ liệu chính — xem ghi chú trên).  
**Cách C — data của bạn**: upload sẵn file `train.jsonl` rồi bỏ qua ô này.


In [ ]:
# ── Cách A: dataset SFT thật (nhẹ, sạch — chạy nhanh trên Kaggle) ──
from datasets import load_dataset
import json

OUT = '/kaggle/working/train.jsonl'
MAX_PER = 4000            # tăng nếu muốn train lâu/kỹ hơn
n = 0
with open(OUT, 'w', encoding='utf-8') as f:
    # no_robots: người viết tay, rất sạch, dạy văn phong tốt (CC BY-NC 4.0)
    for row in load_dataset('HuggingFaceH4/no_robots', split='train', streaming=True):
        msgs = [{'role': m['role'], 'content': m['content']} for m in row['messages']
                if m.get('role') in ('system','user','assistant') and m.get('content')]
        if len(msgs) >= 2:
            f.write(json.dumps({'messages': msgs}, ensure_ascii=False) + '\n'); n += 1
        if n >= MAX_PER: break
print('Đã ghi', n, 'mẫu →', OUT)


In [ ]:
# ── (TÙY CHỌN) Cách B: bake skills LUMINA. Bỏ comment nếu muốn thêm. ──
# SKILLS_DIR = '/kaggle/input/lumina-skills'   # ← Kaggle Dataset bạn upload từ data/skills/
# import os, json
# with open(OUT, 'a', encoding='utf-8') as f:
#     for fn in sorted(os.listdir(SKILLS_DIR)):
#         if not fn.endswith('.md') or fn.upper().startswith('ATTRIBUTION'): continue
#         raw = open(os.path.join(SKILLS_DIR, fn), encoding='utf-8').read()
#         name, desc, body = fn[:-3], '', raw
#         if raw.startswith('---'):
#             end = raw.find('\n---', 3)
#             if end != -1:
#                 fm, body = raw[3:end], raw[end+4:]
#                 for line in fm.splitlines():
#                     if line.startswith('name:'): name = line[5:].strip()
#                     elif line.startswith('description:'): desc = line[12:].strip()
#         body = body.strip()
#         if body:
#             msgs = [{'role':'user','content': f'Hướng dẫn chuyên sâu về: {desc or name}'},
#                     {'role':'assistant','content': body}]
#             f.write(json.dumps({'messages': msgs}, ensure_ascii=False) + '\n')
# print('Đã thêm skills vào', OUT)


## 3) Nạp model 4-bit + cấu hình LoRA

`BASE` mặc định là **Qwen2.5-1.5B-Instruct** — nhẹ nhất mà vẫn khá, file Ollama cuối chỉ **~1GB**.
Muốn nhẹ hơn nữa: `Qwen/Qwen2.5-0.5B-Instruct` (~0.4GB). Muốn mạnh hơn: 3B (~2GB) / 7B (~4GB).
Model 1B–1.5B train **rất nhanh** trên Kaggle free (thừa VRAM). **Đừng** chọn > 8B.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
import torch

BASE = 'Qwen/Qwen2.5-1.5B-Instruct'   # ~1.5B → file Ollama cuối ~1GB, siêu nhẹ (mặc định)
# Nhẹ hơn nữa:  'Qwen/Qwen2.5-0.5B-Instruct'  (~0.4GB — nhẹ nhất, chất lượng thấp hơn)
# Đúng 1B:      'meta-llama/Llama-3.2-1B-Instruct' (cần duyệt license HF) · 'google/gemma-3-1b-it'
# Mạnh hơn:     'Qwen/Qwen2.5-3B-Instruct' (~2GB) · 'Qwen/Qwen2.5-7B-Instruct' (~4GB)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16)
model.config.use_cache = False

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj',
                                  'gate_proj','up_proj','down_proj'])


## 4) Train (QLoRA / SFT)

T4 x2 hợp `bf16`. Nếu OOM: giảm `max_seq_length` (1024), `per_device_train_batch_size`,
hoặc đổi BASE sang 3B. Mặc định 1 epoch — đủ để thấy hiệu ứng.


In [ ]:
from datasets import load_dataset as _ld
from trl import SFTTrainer, SFTConfig

ds = _ld('json', data_files=OUT, split='train')
def fmt(ex):
    return {'text': tok.apply_chat_template(ex['messages'], tokenize=False,
                                            add_generation_prompt=False)}
ds = ds.map(fmt, remove_columns=ds.column_names)

cfg = SFTConfig(output_dir='/kaggle/working/lumina-lora', num_train_epochs=1,
                per_device_train_batch_size=1, gradient_accumulation_steps=16,
                learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.03,
                logging_steps=10, save_strategy='epoch', bf16=True,
                max_seq_length=2048, packing=True, report_to='none',
                gradient_checkpointing=True)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds,
                     peft_config=lora, processing_class=tok)
trainer.train()
trainer.save_model('/kaggle/working/lumina-lora')
tok.save_pretrained('/kaggle/working/lumina-lora')
print('✅ Đã lưu adapter → /kaggle/working/lumina-lora')


## 5) Đóng gói adapter để TẢI VỀ

Sau khi chạy xong: mở tab **Output** bên phải → tải `lumina-lora.zip`. Đây là **LoRA
adapter** (thường 50–400MB) — nhỏ vì chỉ là *phần chênh* học thêm, KHÔNG chứa 7B gốc.


In [ ]:
import shutil, os
shutil.make_archive('/kaggle/working/lumina-lora', 'zip', '/kaggle/working/lumina-lora')
sz = os.path.getsize('/kaggle/working/lumina-lora.zip')/1e6
print(f'lumina-lora.zip = {sz:.1f} MB  (tab Output → Download)')


## 6) Đưa vào LUMINA (chạy trên máy bạn, sau khi tải adapter về)

```bash
# a) Gộp adapter vào base rồi convert GGUF 4-bit cho Ollama (xem export_ollama.md):
#    python -m peft ... merge  →  llama.cpp convert  →  quantize q4_K_M
# b) ollama create lumina-local -f Modelfile.example
# c) .env của LUMINA:
#      OLLAMA_BASE_URL=http://localhost:11434/v1
#      LOCAL_MODELS=lumina-local
#      LOCAL_ONLY=true          # ← 0đ token, KHÔNG cần API key nào của model ngoài
```

Đây chính là **"API LUMINA của riêng bạn, không cần key"**: LUMINA (router + skills)
gọi model bạn tự fine-tune chạy local qua Ollama. API riêng của LUMINA (`/v1/chat/
completions` + key `lum_...`) **đã có sẵn** — nó chỉ cần *một bộ não* phía sau, và bộ
não local này là bộ não free đó.


## ⚠️ Về ý "nén 22B từ 16-bit → 4-bit thành file < 400MB"

**Không thể** — đây là giới hạn vật lý, không phải thiếu kỹ thuật:

| Độ chính xác | 22B nặng bao nhiêu |
|---|---|
| 16-bit (fp16) | ~44 GB |
| 8-bit (int8)  | ~22 GB |
| **4-bit (AWQ/GPTQ/nf4)** | **~11–13 GB** |
| 1.58-bit (ternary, cực đoan) | ~4–5 GB |

400MB cho 22B = ~0,15 bit/tham số — **dưới cả mức ternary**, model sẽ hỏng hoàn toàn.

- **AWQ** ([mit-han-lab/llm-awq](https://github.com/mit-han-lab/llm-awq)) là 4-bit THẬT,
  chất lượng gần như nguyên bản — nhưng 22B AWQ vẫn ~**11–13GB**, KHÔNG phải 400MB.
  Nó để **chạy** model 22B trên 1 GPU 16–24GB, không phải để thu nhỏ xuống 400MB.
- Cái **< 400MB** duy nhất hợp lý là **LoRA adapter** — nhưng nó là *bản vá* cần model
  gốc kèm theo, KHÔNG phải model 22B đứng một mình.
- Muốn *một file nhỏ chạy tốt*: chọn model **nhỏ hơn** (3B–8B) rồi AWQ/GGUF 4-bit →
  file ~2–5GB. Đó là con đường thực tế; ép 22B < 400MB thì không có cách nào.
